In [59]:
# ====================== 环境准备 ======================
# 1) 设定环境变量（必须在导入 matplotlib 之前执行）
import os
from copy import deepcopy
from copy import deepcopy

# from draw.pymatlab.dataresult_mathdology_multi2 import pending_edges

os.environ['QT_API'] = 'pyqt5'        # 指定使用 PyQt5 作为 Qt 绑定
os.environ['MPLBACKEND'] = 'QtAgg'    # 指定 Matplotlib 后端为 QtAgg（更推荐，替代 TkAgg）

# 2) 启动 Qt 事件循环（控制台模式下，让 Qt 窗口能实时响应）
%gui qt5
from copy import deepcopy

# 3) 检查 matplotlib backend
import matplotlib as mpl
mpl.rcParams.update({
    # 这里按系统常见字体给一串候选，存在则自动生效
    "font.sans-serif": ["Microsoft YaHei", "SimHei", "SimSun",
                        "Noto Sans CJK SC", "Source Han Sans SC",
                        "Arial Unicode MS", "DejaVu Sans"],
    "font.family": "sans-serif",
    "axes.unicode_minus": False,   # 负号用正常字符，避免被当作缺字形
})
import basicSa.fileread.readsatellite as readsatellite
import basicSa.simulator.nodemanager as nodemanager
print("backend (before pyplot):", mpl.get_backend())
# 如果不是 QtAgg，强制改为 QtAgg（注意：必须在导入 pyplot 前设置）
mpl.rcParams['backend'] = 'QtAgg'

# 4) 现在再导入 pyplot
import matplotlib.pyplot as plt
print("backend (after pyplot):", mpl.get_backend())

# ====================== 导入依赖 ======================
import sys
# 避免反复执行时 Qt 类重复导入导致崩溃：如果已加载，先删除再导入
if 'draw.pyqt_draw.pyqt_main2' in sys.modules:
    del sys.modules['draw.pyqt_draw.pyqt_main2']

from PyQt5 import QtWidgets
import pyqtgraph as pg
from draw.pyqt_draw.pyqt_main2 import SatelliteViewer
import draw.read_snap_xml as read_snap_xml

# 配置 pyqtgraph：开启抗锯齿，关闭 OpenGL（更稳定）
pg.setConfigOptions(antialias=True)
# pg.setConfigOptions(useOpenGL=False)   # 若驱动或 OpenGL 有问题可显式关闭
# ====================== 基础参数 ======================
# 星座参数：每轨道卫星数 N，轨道平面数 P
N = 36
P = 18

from config import DATA_DIR,INPUT_DIR

from pathlib import Path

backend (before pyplot): QtAgg
backend (after pyplot): QtAgg


In [16]:
#sat_dir_path = r'C:\usrspace\mywork\generic\data\648qianfan_xml'  # raw string for file path
sat_dir_path = r'C:\usrspace\mywork\data\648qianfan1d_xml'  # raw string for file path

#sat_dir_path = '/home/yfh/Desktop/Data/onehun_ecef'
satangle = 45
track_angle = 89

BaseRAAN_INCREMENT = 18



SatelliteManager = nodemanager.SatelliteManager()
readsatellite.readsatellite(SatelliteManager,sat_dir_path, satangle, track_angle, P, N, BaseRAAN_INCREMENT, t_start=0, t_end=22005)


(648, 648, [])

In [17]:
satellite = SatelliteManager.satellites

satellite1=satellite[644]
satellite1[22001]



In [60]:
SIMULATION_EDITION = 'motif3'

In [61]:
from draw.pyqt_draw.pyqt_main2 import SatelliteViewer


In [62]:
time_2_build = 60
TIME_2_BUILD = time_2_build

# 默认用 "topology_{TIME_2_BUILD}"，也允许用环境变量 TOPOLOGY_VERSION 覆盖
DEFAULT_VERSION = f"{SIMULATION_EDITION}/topology_{TIME_2_BUILD}"
VERSION = os.getenv("TOPOLOGY_VERSION", DEFAULT_VERSION)


RAW_DIR    = Path(INPUT_DIR) / VERSION / "raw"

CONFIG_DIR = Path(INPUT_DIR) / f"{SIMULATION_EDITION}/config"
MODIFY_DIR    = Path(INPUT_DIR) / VERSION / "modify"
FIGURE_DIR = Path(INPUT_DIR) / VERSION / "figure"
# 若不存在则创建（递归创建上级目录；已存在不报错）
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

In [63]:
import genaric2.tegnode as tegnode
import draw.basic_functio.write2xml as write2xml

# 这里，我们读取到nodes 信息,但是我们需要转化为edge信息，注意这里主要就只有包括inter-edge信息


In [19]:
import importlib
importlib.reload(read_snap_xml)

<module 'draw.read_snap_xml' from 'C:\\usrspace\\mywork\\generic\\draw\\read_snap_xml.py'>

In [64]:
file_in = DATA_DIR
# xml_file = r"DATA_DIR\station_visible_satellites_648_1d_real.xml"
xml_file = DATA_DIR / "station_visible_satellites_648_1d_real.xml"

def slice_group_data(raw_group_data, start, end):
    """
    从 raw_group_data 中裁剪时间区间 [basicSa, end)
    """
    return {
        step: raw_group_data[step]
        for step in range(start, end)
        if step in raw_group_data
    }
# 只做一次：解析大区间
RAW_START, RAW_END = 0, 22006
raw_group_data = read_snap_xml.parse_xml_group_data(xml_file, RAW_START, RAW_END)


In [65]:
series = read_snap_xml.parse_station_timeseries(xml_file, [0,1,2,11,13,14], RAW_START, RAW_END)


In [66]:


beijin = series[0]
chongqin = series[1]
wulumuqi = series[2]

huasha = series[3]
boling = series[4]
paris = series[5]

In [7]:
import importlib
importlib.reload(write2xml)

True


<module 'draw.basic_functio.write2xml' from 'C:\\usrspace\\mywork\\generic\\draw\\basic_functio\\write2xml.py'>

In [84]:
RANGES = [
    (0,1204),(1204,3669),(3669,4094),(4094,6814),(6814,8485),(8485,11640),
     (11640,13057),(13057,14065),(14065,16604),(16604,18396),
    (18396,19831),(19831,20814),(20814,22005)
]



paths = [MODIFY_DIR /f"interplane_links_{s}_{e}.xml" for s, e in RANGES]

# 方式 A：顺序（内存最低、稳定）
totalnode = write2xml.load_all_nodes_sequential_test(paths, tegnode.tegnode_new)

# raw版本的
# paths = [RAW_DIR /f"interplane_links_{s}_{e}.xml" for s, e in RANGES]
# totalnode = write2xml.load_all_nodes_sequential_test(paths, tegnode.tegnode_new)



In [80]:
RAW_DIR

WindowsPath('C:/usrspace/mywork/generic/data/input/motif3/topology_60/raw')

In [68]:
import basicSa.fileread.readsatellite as readsatellite
import basicSa.simulator.nodemanager as nodemanager

In [13]:
import  importlib
importlib.reload(readsatellite)

<module 'basicSa.fileread.readsatellite' from 'C:\\usrspace\\mywork\\generic\\basicSa\\fileread\\readsatellite.py'>

In [69]:
start_ts =0
end_ts = 22005

In [70]:
group_data = slice_group_data(raw_group_data, start_ts, end_ts)


In [ ]:
rev_group_data,offset = read_snap_xml.modify_group_data(group_data, N=36, groupid=4)



In [85]:

import draw.basic_functio.motif as motif

all_inter_edge,pending_edge,iG_edge = motif.transform_nodes_2_rawedge_test(totalnode, P, N, start_ts, end_ts)

In [46]:
end_ts


22005

In [72]:
# 转化为inter-edge信息后，我们可以通过绘图来初步查看

# ====================== 绘图初始化 ======================
# 1) QApplication 实例（全局唯一）
app = QtWidgets.QApplication.instance() or QtWidgets.QApplication([])

# 2) 确保 viewer 有全局引用，避免 GC 回收导致崩溃
if not hasattr(sys.modules[__name__], "_viewer_list"):
    _viewer_list = []


In [38]:
# 每个 step 的路径都只有 0 -> 37
# if getattr(viewer, "steps", None):
#     path_by_step = {int(s): [0, 72,73] for s in viewer.steps}



In [26]:
# # 先得到一个长度为 (end_ts-start_ts) 的列表
# paths_seq = [[0, 36, 37] for _ in range(start_ts, end_ts)]
# # 转成 {step: path}
# path_by_step = {start_ts + i: p for i, p in enumerate(paths_seq)}

 下面是导出某一时刻的链路信息，

In [34]:
# here we need check its
# d = all_inter_edge[time_idx]  # 当前时间步的映射：int -> set[int]
#
# # 1) 是否存在键=85
# has_key_85 = 85 in d
#
# # 如果要取它的值（不存在则给空集合）
# val_for_85 = d.get(85, set())
#
# # 2) 是否有哪个键的集合里“包含85”（即是否有人指向 85）
# keys_pointing_to_85 = [k for k, s in d.items() if 85 in s]
# has_value_85 = len(keys_pointing_to_85) > 0
#
# # 3) 同时给个简洁打印
# print(f"键=85 是否存在: {has_key_85}")
# print(f"键=85 的值: {val_for_85}")
# print(f"哪些键指向 85: {keys_pointing_to_85}")




键=85 是否存在: True
键=85 的值: {123}
哪些键指向 85: []


下面是导出某一时刻的group状态

In [ ]:
# from typing import Dict, Set, Tuple, Iterable
# import pandas as pd
#
# Adj = Dict[int, Set[int]]                 # 邻接：u -> {v1, v2, ...}
# AllAdj = Dict[int, Adj]                   # 时刻 -> 邻接
#
# def _edges_from_adj(adj: Adj, directed: bool=False) -> Set[Tuple[int,int]]:
#     """
#     把邻接字典转成边集合。
#     - 无向：边一律存成 (min(u,v), max(u,v))，避免重复计数
#     - 有向：边存成 (u, v)
#     """
#     edges: Set[Tuple[int,int]] = set()
#     for u, nbrs in adj.items():
#         if not isinstance(nbrs, Iterable):
#             continue
#         for v in nbrs:
#             if u == v:
#                 continue
#             if directed:
#                 edges.add((u, v))
#             else:
#                 a, b = (u, v) if u < v else (v, u)
#                 edges.add((a, b))
#     return edges
#
# def summarize_topology(all_inter_edge: AllAdj, directed: bool=False) -> pd.DataFrame:
#     """
#     统计每时刻的链路数量与拓扑是否变化（相对前一时刻）。
#     返回列：
#       - t: 时刻
#       - link_count: 边条数
#       - changed: 是否变化（0/1）
#       - add_edges: 本时刻相对上一时刻 新增的边数量
#       - del_edges: 本时刻相对上一时刻 删除的边数量
#     """
#     times = sorted(all_inter_edge.keys())
#     rows = []
#     prev_edges: Set[Tuple[int,int]] = set()
#
#     for i, t in enumerate(times):
#         edges_t = _edges_from_adj(all_inter_edge[t], directed=directed)
#         link_count = len(edges_t)
#
#         if i == 0:
#             changed = 0
#             add_cnt = del_cnt = 0
#         else:
#             add = edges_t - prev_edges
#             rem = prev_edges - edges_t
#             add_cnt, del_cnt = len(add), len(rem)
#             changed = 1 if (add_cnt or del_cnt) else 0
#
#         rows.append({
#             "t": t,
#             "link_count": link_count,
#             "changed": changed,      # 0 = 无变化，1 = 有变化
#             "add_edges": add_cnt,
#             "del_edges": del_cnt,
#         })
#         prev_edges = edges_t
#
#     df = pd.DataFrame(rows)
#     return df
#
# # ===== 示例：用你的 all_inter_edge 运行 =====
# df = summarize_topology(all_inter_edge, directed=False)
# # 查看：print(df.head())
# # 想导出 Origin/CSV：df.to_csv("topology_change_summary.csv", index=False)
# # df.to_csv("topology_change_summary.csv", index=False)

In [86]:
# 3) 创建并配置 viewer
viewer = SatelliteViewer(group_data)
viewer.setWindowTitle("modify ")
viewer.resize(1200, 700)
viewer.edges_by_step = all_inter_edge
viewer.pending_links_by_step=   pending_edge
viewer.IG_link_by_step =iG_edge
viewer.show()



注意，上述的拓扑，是只有异轨链路的，并且，在邻接表上，也是单向的。因此，我们实际上要做这几件事情

1.实际网络拓扑是还有同轨链路的，所以，我们还是要加上同轨链路信息

2.在邻接表上，我们需要将所有的边都转化为双向的

In [76]:
#添加同轨链路

def build_intra_edges_copies(start_ts, end_ts, P, N):
    # 预计算每个节点的左右邻居（tuple 轻量不可变，便于快速构造 set）
    base_neighbors = {
        i * N + j: (i * N + ((j + 1) % N), i * N + ((j - 1) % N))
        for i in range(P) for j in range(N)
    }

    all_intra_edge = {}
    for step in range(start_ts, end_ts):
        # 一次性构造（避免 setdefault & 多次 add 的开销）
        adj = {node: set(neis) for node, neis in base_neighbors.items()}
        all_intra_edge[step] = adj
    return all_intra_edge

# 用法
all_intra_edge = build_intra_edges_copies(start_ts, end_ts, P, N)


In [77]:
# 异轨链路双向化


def make_edges_bidirectional(edge_dict):
    """
    edge_dict: {src: Iterable[dst, ...], ...}
    返回新的 dict[int, set[int]]，不会修改入参
    """
    new_edges = {}
    for src, dsts in edge_dict.items():
        for dst in set(dsts):
            if src == dst:   # 可选：去自环
                continue
            new_edges.setdefault(src, set()).add(dst)
            new_edges.setdefault(dst, set()).add(src)
    return new_edges

# ✅ 生成全新的 used_all_inter_edge（不改 all_inter_edge）
used_all_inter_edge = {
    step: make_edges_bidirectional(edges_t)
    for step, edges_t in all_inter_edge.items()
}


# all_inter_edge=[]
# for step in all_inter_edge:
#     all_inter_edge[step] = make_edges_bidirectional(all_inter_edge[step])


In [52]:
used_all_inter_edge[1].get(72)

{36, 108}

In [78]:
all_edges = {}

for step in range(start_ts, end_ts):
    all_edges[step] = {}
    # 先合并intra_edge
    if step in all_intra_edge:
        for src, dsts in all_intra_edge[step].items():
            all_edges[step].setdefault(src, set()).update(dsts)
    # 再合并inter_edge
    if step in used_all_inter_edge:
        for src, dsts in used_all_inter_edge[step].items():
            all_edges[step].setdefault(src, set()).update(dsts)


In [79]:

viewer = SatelliteViewer(group_data)
viewer.setWindowTitle("main the bidrection of the satellite network")
viewer.resize(1200, 700)
viewer.edges_by_step = all_edges
viewer.pending_links_by_step = pending_edge
viewer.IG_link_by_step =iG_edge
viewer.show()
_viewer_list.append(viewer)

有了所有时间拓扑信息后，我们接下来就可以研究，某些时刻下，某些点之间的最短路径情况。




In [25]:
# from collections import deque
# from typing import Dict, Set, Iterable, List, Optional, Any
#
# Adj = Dict[int, Set[int]]
#
# def make_undirected(adj: Adj) -> Adj:
#     """
#     输入：单步拓扑的邻接表 {u: {v,...}, ...}（可能是有向）
#     输出：新的无向邻接表（不会修改入参）
#     """
#     g: Adj = {}
#     for u, vs in adj.items():
#         if vs is None:
#             continue
#         g.setdefault(int(u), set())
#         for v in vs:
#             if v is None:
#                 continue
#             u2, v2 = int(u), int(v)
#             if u2 == v2:
#                 continue  # 去自环（可选）
#             g.setdefault(u2, set()).add(v2)
#             g.setdefault(v2, set()).add(u2)
#     return g
#
# def bfs_shortest_path(adj: Adj, start: Any, end: Any, *, undirected: bool = True) -> Optional[List[int]]:
#     """
#     在无权图上求最短路（节点/边权都等于1）。
#     - adj: 单步拓扑邻接表 {u: {v,...}, ...}
#     - start, end: 起点/终点（int 或可转 int 的字符串）
#     - undirected: True=按无向图求解；False=按有向边求解
#     返回：最短路径的节点列表，如 [s, ..., t]；不可达返回 None
#     """
#     s = int(start)
#     t = int(end)
#
#     # 准备邻接（是否无向）
#     G = make_undirected(adj) if undirected else {int(u): set(map(int, vs)) for u, vs in adj.items()}
#
#     if s == t:
#         # 起终点相同，视为零长度路径（若希望至少回传 [s]）
#         return [s]
#
#     if s not in G and s not in adj:
#         return None
#     if t not in G and t not in adj:
#         return None
#
#     # 确保起点在字典里（即使它当前没有出边）
#     G.setdefault(s, set())
#
#     # BFS
#     q = deque([s])
#     visited = {s}
#     parent: Dict[int, int] = {}
#
#     while q:
#         u = q.popleft()
#         # 如果某些节点不在 G（仅作为被指向的“孤点”），给它空集合
#         for v in G.get(u, set()):
#             if v in visited:
#                 continue
#             visited.add(v)
#             parent[v] = u
#             if v == t:
#                 # 回溯重建路径
#                 path = [t]
#                 while path[-1] != s:
#                     path.append(parent[path[-1]])
#                 path.reverse()
#                 return path
#             q.append(v)
#
#     return None  # 不可达
#
# def path_to_edges(path: List[int]) -> List[tuple]:
#     """把节点序列转成边序列 [(u0,u1), (u1,u2), ...]"""
#     if not path or len(path) < 2:
#         return []
#     return list(zip(path[:-1], path[1:]))


In [72]:
# # 单步拓扑：all_edges[1] 形如 {src: {dst1, dst2, ...}, ...}
# adj_step1 = all_edges[time_idx]
#
# # 求最短路（无向）
# path = bfs_shortest_path(adj_step1, start=355, end=44, undirected=True)
# print("path:", path)                   # e.g. [12, 47, 118, 345]
# print("edges:", path_to_edges(path))   # e.g. [(12,47), (47,118), (118,345)]


path: [355, 356, 357, 322, 287, 216, 181, 146, 111, 76, 77, 78, 42, 43, 44]
edges: [(355, 356), (356, 357), (357, 322), (322, 287), (287, 216), (216, 181), (181, 146), (146, 111), (111, 76), (76, 77), (77, 78), (78, 42), (42, 43), (43, 44)]


In [33]:
# allpath_length = []
#
# for t in range(start_ts, end_ts):
#     adj = all_edges[t]                      # 若 all_edges 是 dict，可用 all_edges.get(t, {})
#     best = None                             # 当前时刻的最短路径长度
#
#     # 若该时刻没有候选源/宿，直接记为 None 或 inf
#     sources = paris[t]
#     dests   = chongqin[t]
#     if not sources or not dests:
#         allpath_length.append(None)         # 或 float("inf")
#         continue
#
#     for s in sources:
#         for d in dests:
#             path = bfs_shortest_path(adj, start=s, end=d, undirected=True)
#             if path is None:
#                 continue
#             L = len(path)                   # 若你需要“跳数”，通常是 len(path)-1
#             if best is None or L < best:
#                 best = L
#
#     allpath_length.append(best if best is not None else float("inf"))


In [73]:

# t = 899
#
# adj = all_edges[t]                      # 若 all_edges 是 dict，可用 all_edges.get(t, {})
# best = None                             # 当前时刻的最短路径长度
# best_path = None
# # 若该时刻没有候选源/宿，直接记为 None 或 inf
# sources = paris[t]
# dests   = chongqin[t]
#
#
#
# for s in sources:
#     for d in dests:
#         path = bfs_shortest_path(adj, start=s, end=d, undirected=True)
#         if path is None:
#             continue
#         L = len(path)                   # 若你需要“跳数”，通常是 len(path)-1
#         if best is None or L < best:
#             best = L
#             best_path = path
#
# print("best:", best)
# print("best_path:", best_path)


best: 7
best_path: [426, 425, 424, 423, 495, 567, 639]


In [44]:
import  importlib
importlib.reload(plot_2city_shortest_path)

<module 'draw.pymatlab2.chartalgorithm.plot_2city_shortest_path' from 'C:\\usrspace\\mywork\\generic\\draw\\pymatlab2\\chartalgorithm\\plot_2city_shortest_path.py'>

接下来我们就是要求取下列对的最短路径
paris = series[3]
chongqin = series[1]
huasha = series[2]
beijin = series[0]

1. paris-chongqin
2. huasha-beijin

In [58]:
import  draw.pymatlab2.chartalgorithm.plot_2city_shortest_path as plot_2city_shortest_path
# 仅导出 CSV



# df = plot_2city_shortest_path.compute_stationpair_min_hops_over_time(
#     all_edges, paris, chongqin,
#     steps=(0, 1000),
#     undirected=True,
#     return_pair=True,
#     return_path=False  # 如需导出路径节点序列改 True（会更慢一些）
# )

# 导出 CSV（Origin/后续流程用）
csv_path = plot_2city_shortest_path.export_stationpair_min_hops_to_origin(
    all_edges, boling, beijin,
    out_dir=FIGURE_DIR,
    basename=f"minhops_boling_beijin_ttb{TIME_2_BUILD}",
    steps=(0, 10000),
    undirected=True,
    with_pair=True,
    with_path=False
)
print("written:", csv_path)

# # 画图
# plot_2city_shortest_path.plot_stationpair_min_hops_over_time(
#     df,
#     title="Paris–Chongqing minimal shortest hops vs. time",
#     save=True, basename=f"minhops_paris_chongqing_ttb{TIME_2_BUILD}"
# )




written: C:\usrspace\mywork\generic\data\input\motif1\topology_60\figure\minhops_boling_beijin_ttb60.csv


In [56]:
FIGURE_DIR

WindowsPath('C:/usrspace/mywork/generic/data/input/motif1/topology_60/figure')

In [38]:
import  draw.pymatlab2.chartalgorithm.plot_2city_shortest_path as plot_2city_shortest_path

df = plot_2city_shortest_path.compute_stationpair_min_hops_over_time(
    all_edges, boling, beijin,
    steps=(start_ts, end_ts),#, undirected=undirected,
 #   return_pair=with_pair,
       return_path=True
)


In [39]:
df['path'][1]

'431->430->429->428->427->389->351->313->275->237->199->161->160->159->158->157'

In [40]:

#这串代码主要是将上述的端到端最短路径的path进行转化，用于二位显示而已
import re
import pandas as pd

TOTAL_SATS = 36 * 18  # 你的全局里已经有这个，保持一致

def parse_path_str(s: str):
    if pd.isna(s):
        return None
    parts = re.split(r'\s*->\s*', str(s).strip())
    try:
        arr = [int(p) for p in parts if p != '']
    except ValueError:
        return None
    return arr if len(arr) >= 2 else None

def df_paths_to_dict(df: pd.DataFrame, col_path='path', step_col: str | None = 'time',
                     total_sats: int | None = TOTAL_SATS):
    """
    把 df[col_path] 解析成 {step: [node_ids...]}。
    - step_col 存在就用它；否则用 df.index。
    - total_sats 不为 None 时会把节点ID限制到 [0, total_sats).
    """
    path_by_step = {}
    if step_col and step_col in df.columns:
        it = df[[step_col, col_path]].itertuples(index=False, name=None)
        for step, s in it:
            path = parse_path_str(s)
            if path is None:
                continue
            if total_sats is not None:
                path = [v for v in path if 0 <= v < total_sats]
            if len(path) >= 2:
                path_by_step[int(step)] = path
    else:
        for step, s in df[col_path].items():
            path = parse_path_str(s)
            if path is None:
                continue
            if total_sats is not None:
                path = [v for v in path if 0 <= v < total_sats]
            if len(path) >= 2:
                path_by_step[int(step)] = path
    return path_by_step


In [41]:
# 得到可以用于二维显示的数据
path_by_step = df_paths_to_dict(df, col_path='path', step_col='time')


接下来就是时间片的路由显示了

In [42]:
viewer = SatelliteViewer(group_data)
viewer.setWindowTitle("main the bidrection of the satellite network")
viewer.resize(1200, 700)
viewer.edges_by_step = all_edges
viewer.pending_links_by_step = pending_edge
viewer.IG_link_by_step =iG_edge

viewer.set_paths(path_by_step)

viewer.show()


In [ ]:

import draw.step1.function.linkjson as linkjson

out_json = r"C:\usrspace\mywork\generic\data\snapshot_100.json"

In [51]:
time_idx = 998

In [52]:

active = []

# 遍历 all_inter_edge[0]
for source, targets in all_inter_edge[time_idx].items():
    for target in targets:
        # 为每个 link 创建一个字典并追加到 links 列表
        active.append((source,  target))
as_str=True
for x in range(P):
    for y in range(N):
        curr = x * N + y
        nxt  = x * N + ((y + 1) % N)
        # 防守式判断，确保字典里确有这些编号的卫星
        if curr in SatelliteManager.satellites and nxt in SatelliteManager.satellites:
            s = str(curr) if as_str else curr
            t = str(nxt)  if as_str else nxt
            active.append((  s,   t))

# 遍历 all_inter_edge[0]
pending = []
edge_at_t = pending_edge.get(time_idx)  # 如果没有 key，返回 None
if edge_at_t:                            # None / 空字典 都不会进来
    for source, targets in edge_at_t.items():
        for target in targets:
            pending.append((source, target))

# active = [(0,1), (1,2), {"source":2, "target":3}]
# pending = [(3,4), {"source":5, "target":6}]
linkjson.export_satellites_snapshot_to_json(SatelliteManager, time_idx, out_json,
                                   links_active=active, links_pending=pending)


'C:\\usrspace\\mywork\\generic\\data\\snapshot_100.json'

下面是group的输出

In [53]:
# 选择导出的分组
selected_groups = [0, 4]
out_path = r"C:\usrspace\mywork\generic\data\highlighted.json"

# 每个分组的颜色
color_map = {
    0: "#ff0000",  # 红色
    1: "#00ff00",  # 绿色
    2: "#0000ff",  # 蓝色
    3: "#FFA500",  # 紫色
    4: "#800080",  # 黄色
    5: "#00FFFF",  # 橙色
    6: "#FFFF00"   # 青色
}



# 调用函数并保存文件
exported_path = linkjson.export_highlighted_to_json(group_data[time_idx], selected_groups, color_map, out_path)

In [54]:
# 下面就是路径了
path_by_step[time_idx]

[461,
 460,
 459,
 458,
 420,
 382,
 344,
 306,
 268,
 230,
 192,
 191,
 190,
 189,
 188,
 187]

In [ ]:
for node in path_by_step[time_idx]

接下来，我们可以查看在这段时间内，平均最短路径的变化情况


In [ ]:
# 1) 只在 Jupyter 里非阻塞显示
import  draw.pymatlab2.chartalgorithm.plot_intergroup_avg_shortest_path as plot_intergroup_avg_shortest_path
# plot_intergroup_avg_shortest_path.plot_intergroup_avg_shortest_path(all_edges, group_data, group_a=0, group_b=4)

# 假设已有 all_edges, group_data
csv_path = plot_intergroup_avg_shortest_path.export_intergroup_avgspath_to_origin(
    all_edges, group_data,
    out_dir=FIGURE_DIR,
    basename=f"avgspath_g0_4_ttb{TIME_2_BUILD}",
    group_a=0, group_b=4,
    steps=(0, 22005),         # 或 None / 自定义迭代器
    undirected=True
)
print("written:", csv_path)


# # 2) 显示 + 保存 PNG/PDF
# plot_intergroup_avg_shortest_path(
#     all_edges, group_data,
#     group_a=0, group_b=4,
#     save=True,                 # 用默认保存规则
#     save_dir=FIGURE_DIR,       # 你的 figure 目录
#     basename="avgspath_g0_g4",
#     formats=("png","pdf"),
#     dpi=300
# )
#
# # 3) 不显示只保存，拿到 DataFrame 做后续处理
# fig, ax, df_metric = plot_intergroup_avg_shortest_path.plot_intergroup_avg_shortest_path(
#     all_edges, group_data,
#     save_dir=FIGURE_DIR,
#     show=False,
#     save=["avgspath.png", "avgspath.pdf"]
# )


In [ ]:
FIGURE_DIR

In [ ]:
import  importlib
importlib.reload(plot_switches_nonblocking)

1. 写成一个专门的函数 2. 在jupter里运行的时候，能够非阻塞 3.我能够选择是否保存图片和pdf

In [ ]:
##这个代码是显示,每时刻正在建链的链路条数
# 目前已经已经实现
#1 . 直接运行显示
#2.数据输出csv输出交由origin处理


# 平均切换条数显示
import  draw.pymatlab2.chartalgorithm.plot_switches_nonblocking as plot_switches_nonblocking
# 生成切换图
# 1) 只在 Jupyter 里“非阻塞显示”，不保存
# plot_switches_nonblocking.plot_pending_edges_timeseries(pending_edges)

# 2) 显示 + 同时保存 PNG 和 PDF（论文友好）

# plot_switches_nonblocking.plot_pending_edges_timeseries(
#     pending_edges,
#     save=True,
#     save_dir=FIGURE_DIR,
#     basename="pending_30s_4k",
#     formats=("png","svg","pdf"),
#     dpi=300,
#     save_pixels=(3840, 2160),   # ★ 4K
#     keep_display_size=True
# )


# # 3) 不显示、只保存到指定路径（多格式）
# plot_pending_edges_timeseries(
#     pending_edges,
#     show=False,
#     save=["out/pending_edges.png", "out/pending_edges.pdf"]
# )
#
# # 4) 获取返回的 DataFrame，后续自定义处理
# fig, ax, df_counts = plot_pending_edges_timeseries(pending_edges, return_handles=True)
# df_counts.head()
#

#导出
# 你已有：FIGURE_DIR = Path(INPUT_DIR) / VERSION / "figure"
# 没有就先建


# 导出 CSV（Origin 里：数据 → 从文件导入 → 单一 ASCII）
paths = plot_switches_nonblocking.export_pending_series_to_origin(
    pending_edges,
    out_dir=FIGURE_DIR,
    basename="pending_edges_ttb60",
    to=("csv",),            # 也可 ("csv","xlsx")
    fill_missing=True,
    max_t_seconds=None,
    smooth_window=None         # 想要多一列移动平均就填窗口大小；不需要就设 None
)
print(paths)


In [ ]:
FIGURE_DIR

下面是计算平均切换条数

In [ ]:
def average_switches(pending_edges):
    total_edges = 0
    total_steps = 0

    for step, edge_dict in pending_edges.items():
        # edge_dict: dict[src_id] -> set(dst_ids)
        step_edges = sum(len(dsts) for dsts in edge_dict.values())
        total_edges += step_edges
        total_steps += 1

    if total_steps == 0:
        return 0.0

    return total_edges / total_steps

avg = average_switches(pending_edges)
print(f"平均切换跳数: {avg:.2f}")


In [ ]:
# 计算平均切换数（全范围）
import  draw.pymatlab2.chartalgorithm.plot_pending_edges_timeseries as plot_pending_edges_timeseries
avg = plot_pending_edges_timeseries.average_switches(pending_edges)
print(f"平均切换数: {avg:.2f}")

# 在 Jupyter 非阻塞显示，并标注平均值；同时保存 PNG/PDF 到 FIGURE_DIR
fig, ax, df_counts, avg2 = plot_pending_edges_timeseries.plot_pending_edges_timeseries(
    pending_edges,
    max_t_seconds=None,       # 或 22000
    annotate_avg=True,
    save=True,
    save_dir=FIGURE_DIR,      # 你之前定义的目录
    basename="pending_30s",
    formats=("png","pdf"),
    dpi=300
)
